# Database-Driven AcroForm Fill — End-to-End

This notebook walks through filling an editable (AcroForm) PDF from a canonical database JSON, end-to-end. Every intermediate artifact is written to `output/` so the steps are inspectable.

**Python:** tested on **3.12**, requires **>= 3.10**.

**Strategy:** widget extraction follows the same vision-LLM mapping pattern as [`mapped_acroform_demo.ipynb`](mapped_acroform_demo.ipynb) — robust label inference across forms with varying tooltip quality, cached by form fingerprint. The DB-driven fill layer is the new part.

**Standalone behavior:** every algorithmic step is inlined. The only `claims_parser` imports are pydantic data models (so the JSON artifacts stay typed) and the mapped-AcroForm writer (~200 lines of pymupdf widget logic).

Pipeline:

1. **Detect** the PDF carries AcroForm widgets.
2. **Extract** the widget catalog (pymupdf, no labels yet).
3. **OCR** the form with Azure Document Intelligence.
    - **3a.** S0 single-call alternative (preferred on >2-page forms).
4. **Compute** the form fingerprint over the widget catalog.
5. **Map** widgets → labels with the vision LLM.
6. **Build** the generic `FormSchema` from the mapping.
7. **Persist** the mapping under its fingerprint (cache).
8. **Load** the canonical DB JSON (envelope strip + `DBEnvelope` validation).
    - **8a.** (Optional) LLM-fill empty fields in the DB template.
9. **Canonical DB paths** + DB-schema fingerprint (form-independent).
10. **Schema fingerprint** (form-specific cache key for the fillmap).
11. **LLM resolve** — structured-output batch over every field.
12. **Persist** the `FieldMap` (cache).
13. **Materialize** — apply `Resolution`s + DB → `FilledFormSchema`.
14. **Write** values into AcroForm widgets (`write_mapped`).
15. **Summary**.

Change `INPUT_PDF` and `DB_TEMPLATE` in the setup cell to demo a different form or case.

## 0. Setup

All paths are relative to the repo root. The cell below walks up the filesystem until it finds `claims_parser/`, then `chdir`s there. Secrets (Azure DI + OpenAI keys) live in `parser.env`.

In [ ]:
import os, sys
from pathlib import Path

assert sys.version_info >= (3, 10), 'This notebook requires Python 3.10+'

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'claims_parser').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print('Repo root:', REPO_ROOT)
print('Python   :', sys.version.split()[0])

# --- inputs ---
INPUT_PDF    = Path('input/AETNA_form2.pdf')            # change me
DB_TEMPLATE  = Path('database/database_schema.json')    # change me

# --- outputs ---
OUT_DIR = Path('output/intermediate')
OUT_DIR.mkdir(parents=True, exist_ok=True)
Path('output/final').mkdir(parents=True, exist_ok=True)
STEM = INPUT_PDF.stem

# mapped-AcroForm extraction artifacts
WIDGETS_JSON   = OUT_DIR / f'{STEM}.widgets.json'
CONTEXT_JSON   = OUT_DIR / f'{STEM}.context.json'
MAPPING_JSON   = OUT_DIR / f'{STEM}.mapping.json'
SCHEMA_JSON    = OUT_DIR / f'{STEM}.schema.json'

# db-driven fill artifacts
DB_FILLED_JSON = Path('database') / f'{INPUT_PDF.stem}.case.json'
FILLMAP_JSON   = OUT_DIR / f'{STEM}.fillmap.json'
FILLED_JSON    = OUT_DIR / f'{STEM}.filled.json'
FILLED_PDF     = Path('output/final') / f'{STEM}.filled.pdf'
REVIEW_JSON    = OUT_DIR / f'{STEM}.review.json'

print('Input PDF :', INPUT_PDF)
print('DB template:', DB_TEMPLATE)

### Tech stack used by this notebook

- **pymupdf** — widget enumeration + AcroForm field updates.
- **azure-ai-documentintelligence** — `prebuilt-layout` OCR for label inference.
- **openai** — `client.chat.completions.parse(response_format=PydanticModel)` for vision-LLM widget→label mapping and field-resolution (gpt-5-mini).
- **pydantic v2** — typed JSON artifacts at every step.
- **python-dotenv** — loads keys from `parser.env`.

In [ ]:
import json, hashlib, base64, re
from datetime import datetime, timezone
from collections import Counter
from importlib import metadata as im
from typing import Any, Optional

import pymupdf
import openai
from openai import OpenAI
import pydantic
import dotenv
import azure.ai.documentintelligence as azure_di
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import DocumentAnalysisFeature
from azure.core.credentials import AzureKeyCredential

# Pydantic models from the project — kept as imports so JSON artifacts stay typed.
from claims_parser.widget_models  import Widget, WidgetCatalog, WidgetRect
from claims_parser.models         import (
    DocumentContext, PageMetadata, TextLine, KVP, Table, TableCell, SelectionMark,
)
from claims_parser.mapping_models import WidgetBinding, WidgetMapping, _LLMChunkResponse
from claims_parser.schema_models  import FormSchema, FormField, FieldType
from claims_parser.filler_models  import FilledFormField, FilledFormSchema
from claims_parser.review_models  import ReviewItem, ReviewReport
from claims_parser.mapping_cache_models import CachedMapping, CacheIndex, CacheIndexEntry
from claims_parser.db_template_models  import DBEnvelope
from claims_parser.db_mapping_models import (
    Resolution, OptionResolution, FieldResolution, FieldMap,
    DerivedSpec, LLMResolverResponse,
)

# Writer is imported (not inlined): ~200 LOC of widget-update plumbing.
from claims_parser.mapped_writer import write_mapped

dotenv.load_dotenv(REPO_ROOT / 'parser.env')
AZURE_ENDPOINT = os.environ['AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT']
AZURE_KEY      = os.environ['AZURE_DOCUMENT_INTELLIGENCE_KEY']
OPENAI_API_KEY = os.environ['OPENAI_API_KEY']

print(f'pymupdf                       : {pymupdf.__version__}')
print(f'openai                        : {openai.__version__}')
print(f'pydantic                      : {pydantic.VERSION}')
print(f'python-dotenv                 : {im.version("python-dotenv")}')
print(f'azure-ai-documentintelligence : {im.version("azure-ai-documentintelligence")}')
print('Azure endpoint                :', AZURE_ENDPOINT)

## 1. Detect — is this PDF editable?

Open with pymupdf and check whether any page has AcroForm widgets. Only AcroForm PDFs flow through this notebook.

In [ ]:
def is_acroform_pdf(pdf_path: Path) -> bool:
    doc = pymupdf.open(pdf_path)
    try:
        for page in doc:
            if any(True for _ in page.widgets()):
                return True
        return False
    finally:
        doc.close()

print('is_acroform_pdf:', is_acroform_pdf(INPUT_PDF))
print('branch         :', 'acroform' if is_acroform_pdf(INPUT_PDF) else 'scanned')

## 2. Extract the widget catalog

Pure pymupdf — no LLM, no Azure. Iterate every page, every widget, capture field name, type (text/checkbox/radio/dropdown), page, rect, choice values, on-value, and PDF xref. Radio-group options are emitted as separate widgets — that's how PDF AcroForms model them.

In [ ]:
WIDGET_TYPE_MAP = {
    'text':        'text',
    'checkbox':    'checkbox',
    'radiobutton': 'radio',
    'listbox':     'choice',
    'combobox':    'choice',
    'signature':   'signature',
    'pushbutton':  'button',
}

def _on_value(w) -> Optional[str]:
    try:
        v = w.on_state()
    except Exception:
        v = None
    return v or None

def extract_widget_catalog(pdf_path: Path) -> WidgetCatalog:
    doc = pymupdf.open(pdf_path)
    try:
        page_sizes, widgets = [], []
        for page_idx, page in enumerate(doc, start=1):
            page_sizes.append((page.rect.width, page.rect.height))
            for w in page.widgets() or []:
                wt = WIDGET_TYPE_MAP.get((w.field_type_string or '').lower(), 'text')
                r = w.rect
                widgets.append(Widget(
                    field_name=w.field_name or f'_unnamed_{w.xref}',
                    widget_type=wt,
                    rect=WidgetRect(page=page_idx, x0=r.x0, y0=r.y0, x1=r.x1, y1=r.y1),
                    xref=w.xref,
                    tu_label=(w.field_label or None),
                    choice_values=(list(w.choice_values) if getattr(w, 'choice_values', None) else None),
                    on_value=_on_value(w) if wt in ('checkbox', 'radio') else None,
                ))
        return WidgetCatalog(
            file_name=pdf_path.name,
            page_count=doc.page_count,
            page_sizes_pt=page_sizes,
            widgets=widgets,
        )
    finally:
        doc.close()

catalog = extract_widget_catalog(INPUT_PDF)
WIDGETS_JSON.write_text(catalog.model_dump_json(indent=2))

print(f'Pages   : {catalog.page_count}')
print(f'Widgets : {len(catalog.widgets)}')
for t, c in Counter(w.widget_type for w in catalog.widgets).most_common():
    print(f'  {t:10s} {c}')
print(f'Saved   : {WIDGETS_JSON}')

## 3. OCR the form with Azure Document Intelligence

Calls Azure's `prebuilt-layout` model with the key-value-pairs feature. F0 tier caps each request at 2 pages, so we chunk the PDF and stitch results back together with page-number offsets.

On forms > 2 pages, prefer cell 3a (S0 single-call) — older pymupdf versions hit an `IndexError` in `Document.insert_pdf` when chunking AcroForm sources past page 2.

In [ ]:
AZURE_MODEL = 'prebuilt-layout'
AZURE_CHUNK_PAGES = 2

def _polygon_to_list(polygon):
    if not polygon: return []
    first = polygon[0]
    if hasattr(first, 'x') and hasattr(first, 'y'):
        return [c for pt in polygon for c in (pt.x, pt.y)]
    return list(polygon)

def _first_region(regions):
    if not regions: return None, None
    r = regions[0]
    return getattr(r, 'page_number', None), (_polygon_to_list(getattr(r, 'polygon', None)) or None)

def _chunk_pdf_bytes(src_path: Path, start: int, end_inclusive: int) -> bytes:
    sub, src = pymupdf.open(), pymupdf.open(src_path)
    try:
        sub.insert_pdf(src, from_page=start, to_page=end_inclusive)
        return sub.tobytes()
    finally:
        sub.close(); src.close()

def _ingest(result, page_offset: int):
    pages, lines, marks = [], [], []
    for page in result.pages or []:
        actual = (page.page_number or 1) + page_offset
        pages.append(PageMetadata(page_number=actual, width=page.width or 0.0,
                                  height=page.height or 0.0, unit=page.unit or 'inch',
                                  line_count=len(page.lines or [])))
        for ln in page.lines or []:
            lines.append(TextLine(text=ln.content, page=actual,
                                  polygon=_polygon_to_list(ln.polygon)))
        for m in getattr(page, 'selection_marks', None) or []:
            marks.append(SelectionMark(page=actual,
                                       state=getattr(m, 'state', 'unselected') or 'unselected',
                                       polygon=_polygon_to_list(m.polygon),
                                       confidence=getattr(m, 'confidence', None)))
    tables = []
    for tbl in result.tables or []:
        tpage, _ = _first_region(getattr(tbl, 'bounding_regions', None))
        cells = []
        for c in tbl.cells:
            cpage, cpoly = _first_region(getattr(c, 'bounding_regions', None))
            cells.append(TableCell(row=c.row_index, column=c.column_index, text=c.content or '',
                                   page=(cpage or tpage or 1) + page_offset, polygon=cpoly,
                                   row_span=c.row_span or 1, column_span=c.column_span or 1,
                                   kind=getattr(c, 'kind', None)))
        tables.append(Table(page=(tpage or 1) + page_offset,
                            row_count=tbl.row_count, column_count=tbl.column_count, cells=cells))
    kvps = []
    for kvp in result.key_value_pairs or []:
        kpage, kpoly = _first_region(getattr(kvp.key, 'bounding_regions', None))
        vpage, vpoly, vtext = None, None, None
        if kvp.value is not None:
            vpage, vpoly = _first_region(getattr(kvp.value, 'bounding_regions', None))
            vtext = kvp.value.content
        kvps.append(KVP(key_text=kvp.key.content,
                        key_page=(kpage + page_offset) if kpage else None, key_polygon=kpoly,
                        value_text=vtext,
                        value_page=(vpage + page_offset) if vpage else None, value_polygon=vpoly,
                        confidence=getattr(kvp, 'confidence', None)))
    return pages, lines, tables, kvps, marks, result.content or ''

def extract_document_context(pdf_path: Path) -> DocumentContext:
    client = DocumentIntelligenceClient(endpoint=AZURE_ENDPOINT,
                                        credential=AzureKeyCredential(AZURE_KEY))
    total = pymupdf.open(pdf_path).page_count
    all_pages, all_lines, all_tables, all_kvps, all_marks, parts = [], [], [], [], [], []
    for start in range(0, total, AZURE_CHUNK_PAGES):
        end = min(start + AZURE_CHUNK_PAGES - 1, total - 1)
        print(f'  Azure chunk: pages {start+1}-{end+1}')
        poller = client.begin_analyze_document(
            model_id=AZURE_MODEL,
            body=_chunk_pdf_bytes(pdf_path, start, end),
            features=[DocumentAnalysisFeature.KEY_VALUE_PAIRS],
            content_type='application/octet-stream',
        )
        p, l, t, k, m, c = _ingest(poller.result(), start)
        all_pages += p; all_lines += l; all_tables += t
        all_kvps  += k; all_marks += m; parts.append(c)
    return DocumentContext(
        file_name=pdf_path.name, file_path=str(pdf_path), model_id=AZURE_MODEL,
        full_text='\n'.join(parts), pages=all_pages, lines=all_lines,
        tables=all_tables, key_value_pairs=all_kvps, selection_marks=all_marks,
    )

doc_ctx = extract_document_context(INPUT_PDF)
CONTEXT_JSON.write_text(doc_ctx.model_dump_json(indent=2))

print(f'Pages           : {len(doc_ctx.pages)}')
print(f'Lines           : {len(doc_ctx.lines)}')
print(f'Key-value pairs : {len(doc_ctx.key_value_pairs)}')
print(f'Tables          : {len(doc_ctx.tables)}')
print(f'Selection marks : {len(doc_ctx.selection_marks)}')
print(f'Saved           : {CONTEXT_JSON}')

## 3a. Alternative — single-call Azure DI on S0 tier

Cell 10 splits the PDF into 2-page segments for the F0 tier. On S0+, the entire PDF can be analyzed in one call — and as a side benefit this cell sidesteps a known bug in older pymupdf where `insert_pdf` with `from_page>0` raises `IndexError` on AcroForm sources.

**Use this cell when:** your Azure DI resource is S0+ AND your PDF has > 2 pages (or your environment ships an older pymupdf).

**Requires in `parser.env`:** `AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT_S0` and `AZURE_DOCUMENT_INTELLIGENCE_KEY_S0`.

In [ ]:
# Walk up looking for a parser.env carrying the S0 vars.
for _cand in [REPO_ROOT / 'parser.env',
              REPO_ROOT.parent / 'parser.env',
              REPO_ROOT.parent.parent / 'parser.env',
              REPO_ROOT.parent.parent.parent / 'parser.env']:
    if _cand.exists():
        dotenv.load_dotenv(_cand, override=False)

AZURE_ENDPOINT_S0 = os.environ.get('AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT_S0')
AZURE_KEY_S0      = os.environ.get('AZURE_DOCUMENT_INTELLIGENCE_KEY_S0')
if not (AZURE_ENDPOINT_S0 and AZURE_KEY_S0):
    raise RuntimeError(
        'S0 credentials not found. Add AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT_S0 '
        'and AZURE_DOCUMENT_INTELLIGENCE_KEY_S0 to parser.env, or skip this cell '
        'and use cell 10 (chunked F0 path) instead.'
    )

def extract_document_context_single_call(pdf_path: Path) -> DocumentContext:
    client = DocumentIntelligenceClient(
        endpoint=AZURE_ENDPOINT_S0,
        credential=AzureKeyCredential(AZURE_KEY_S0),
    )
    pdf_bytes = pdf_path.read_bytes()                     # no pymupdf in this path
    total = pymupdf.open(pdf_path).page_count             # read-only, version-safe
    print(f'  Azure call (S0, single shot): pages 1-{total}, '
          f'{len(pdf_bytes)/1024:.1f} KB')

    poller = client.begin_analyze_document(
        model_id=AZURE_MODEL,
        body=pdf_bytes,
        features=[DocumentAnalysisFeature.KEY_VALUE_PAIRS],
        content_type='application/octet-stream',
    )
    result = poller.result()
    pages, lines, tables, kvps, marks, content = _ingest(result, page_offset=0)
    return DocumentContext(
        file_name=pdf_path.name, file_path=str(pdf_path), model_id=AZURE_MODEL,
        full_text=content, pages=pages, lines=lines,
        tables=tables, key_value_pairs=kvps, selection_marks=marks,
    )

# Overwrites doc_ctx + CONTEXT_JSON from cell 10 if that was run earlier.
doc_ctx = extract_document_context_single_call(INPUT_PDF)
CONTEXT_JSON.write_text(doc_ctx.model_dump_json(indent=2))

print(f'Pages           : {len(doc_ctx.pages)}')
print(f'Lines           : {len(doc_ctx.lines)}')
print(f'Key-value pairs : {len(doc_ctx.key_value_pairs)}')
print(f'Tables          : {len(doc_ctx.tables)}')
print(f'Selection marks : {len(doc_ctx.selection_marks)}')
print(f'Saved           : {CONTEXT_JSON}')

## 4. Compute the form fingerprint

SHA-256 over a canonical JSON of the widget catalog: widgets sorted by `(page, field_name, on_value, x0, y0)`; rects rounded to ~1pt buckets; `xref` and `tu_label` excluded (not stable across PDF copies). Two users filling the same blank form produce the same fingerprint.

In [ ]:
FINGERPRINT_VERSION = 1
RECT_BUCKET_PT = 1.0

def _bucket(v: float) -> float:
    return round(v / RECT_BUCKET_PT) * RECT_BUCKET_PT

def compute_form_fingerprint(catalog: WidgetCatalog) -> str:
    widgets_sorted = sorted(
        catalog.widgets,
        key=lambda w: (w.rect.page, w.field_name, w.on_value or '',
                       _bucket(w.rect.x0), _bucket(w.rect.y0)),
    )
    payload = {
        'v': FINGERPRINT_VERSION,
        'page_count': catalog.page_count,
        'page_sizes_pt': [[_bucket(w), _bucket(h)] for (w, h) in catalog.page_sizes_pt],
        'widgets': [
            {
                'field_name': w.field_name,
                'widget_type': w.widget_type,
                'page': w.rect.page,
                'on_value': w.on_value,
                'choice_values': list(w.choice_values) if w.choice_values else None,
                'rect': [_bucket(w.rect.x0), _bucket(w.rect.y0),
                         _bucket(w.rect.x1), _bucket(w.rect.y1)],
            }
            for w in widgets_sorted
        ],
    }
    blob = json.dumps(payload, sort_keys=True, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(blob).hexdigest()

fingerprint = compute_form_fingerprint(catalog)
print('Fingerprint:', fingerprint)

## 5. Map widgets → labels (vision LLM)

The expensive step. For each chunk of ≤4 pages we render a high-DPI PNG, pack the widget rects + Azure OCR hints (KVPs, text lines, table cells, selection marks) into a JSON payload, and call `gpt-5-mini` with a strict system prompt. `client.chat.completions.parse(response_format=_LLMChunkResponse)` gives us a structured-output guarantee. Each widget ends up either in `bindings` (with a label and confidence) or in `unmapped_widget_field_names`.

In [ ]:
MAPPER_MODEL          = 'gpt-5-mini'
MAPPER_CHUNK_PAGES    = 4
MAPPER_RENDER_DPI     = 144
MAPPER_SPATIAL_TOL_PT = 100.0

MAPPER_SYSTEM_PROMPT = '''You bind every fillable AcroForm widget on a PDF page to the printed label that
names it on the form.

INPUTS PER CALL
- One or more page images: the authoritative visual rendering of the pages.
- A DI hint set: labels the layout OCR detected on these pages — KVPs,
  text lines, table cells, selection marks. Each carries a polygon in PDF points.
- A widget catalog: every input field on these pages with its rect in PDF
  points, widget_type, opaque PDF field_name, optional author-set tu_label,
  on_value, choice_values, and xref.

COORDINATE SYSTEM
- Both page images and PDF polygons use top-left origin with y growing
  downward. All measurements are in PDF points (1 pt = 1/72 inch).

OUTPUT CONTRACT
- Emit one WidgetBinding per widget OR list the widget's field_name in
  `unmapped_widget_field_names`. Every widget MUST appear in exactly one
  of the two outputs.
- label_text: the printed label, copied verbatim.
- label_source: "kvp", "text_line", "table_cell", or "image_only".
- label_polygon: copy verbatim from the DI hint. Null ONLY when image_only.
- label_page MUST equal the widget's rect page. Never bind across pages.
- semantic_field_id: lowercase snake_case derived from the label. Radio
  siblings sharing a label share this id.
- option_label: for radio siblings, the choice text adjacent to THIS widget
  (e.g. "Yes" / "No"). Otherwise null.
- confidence: 0..1. reasoning: one short sentence.

HARD RULES
1. Every widget MUST end in bindings or unmapped_widget_field_names.
2. Never paraphrase label_text. Copy verbatim.
3. Never invent a label that is not present on THIS form's image.
4. If no plausible printed label is visible near a widget, mark it unmapped.
'''

def _render_page_b64(doc, page_idx: int, dpi: int) -> str:
    return base64.b64encode(doc[page_idx].get_pixmap(dpi=dpi).tobytes('png')).decode('ascii')

def _di_for_pages(ctx: DocumentContext, pages: set) -> dict:
    return {
        'key_value_pairs': [k.model_dump() for k in ctx.key_value_pairs if k.key_page in pages],
        'text_lines':      [l.model_dump() for l in ctx.lines if l.page in pages],
        'table_cells':     [c.model_dump() for t in ctx.tables for c in t.cells
                            if c.page in pages and c.text.strip()],
        'selection_marks': [m.model_dump() for m in ctx.selection_marks if m.page in pages],
    }

def _rect_label_distance(rect, polygon: list) -> float:
    xs, ys = polygon[0::2], polygon[1::2]
    lx0, lx1 = min(xs), max(xs); ly0, ly1 = min(ys), max(ys)
    dx = max(rect.x0 - lx1, lx0 - rect.x1, 0.0)
    dy = max(rect.y0 - ly1, ly0 - rect.y1, 0.0)
    return (dx * dx + dy * dy) ** 0.5

def _validate_chunk(parsed, chunk_widgets):
    by_xref = {w.xref: w for w in chunk_widgets}
    by_name = {w.field_name: w for w in chunk_widgets}
    seen, valid = set(), []
    unmapped = list(parsed.unmapped_widget_field_names)
    for b in parsed.bindings:
        w = by_xref.get(b.widget_xref) or by_name.get(b.widget_field_name)
        if w is None or w.xref in seen: continue
        if b.label_page != w.rect.page:
            unmapped.append(w.field_name); seen.add(w.xref); continue
        if b.label_source != 'image_only' and b.label_polygon:
            if _rect_label_distance(w.rect, b.label_polygon) > MAPPER_SPATIAL_TOL_PT:
                b = b.model_copy(update={'confidence': max(0.0, b.confidence - 0.2)})
        b = b.model_copy(update={
            'widget_xref': w.xref, 'widget_field_name': w.field_name,
            'on_value': w.on_value if w.widget_type in ('checkbox', 'radio') else None,
        })
        valid.append(b); seen.add(w.xref)
    for w in chunk_widgets:
        if w.xref not in seen and w.field_name not in unmapped:
            unmapped.append(w.field_name)
    return valid, list(dict.fromkeys(unmapped))

def map_widgets(pdf_path: Path, ctx: DocumentContext, catalog: WidgetCatalog,
                model: str = MAPPER_MODEL) -> WidgetMapping:
    client = OpenAI(api_key=OPENAI_API_KEY)
    widgets_by_page = {}
    for w in catalog.widgets:
        widgets_by_page.setdefault(w.rect.page, []).append(w)

    all_bindings, all_unmapped, chunks = [], [], 0
    doc = pymupdf.open(pdf_path)
    try:
        for start in range(1, catalog.page_count + 1, MAPPER_CHUNK_PAGES):
            pages = list(range(start, min(start + MAPPER_CHUNK_PAGES, catalog.page_count + 1)))
            chunk_widgets = [w for p in pages for w in widgets_by_page.get(p, [])]
            if not chunk_widgets: continue
            chunks += 1

            text_payload = {
                'pages': pages,
                'page_sizes_pt': {p: catalog.page_sizes_pt[p - 1] for p in pages},
                'widgets': [w.model_dump() for w in chunk_widgets],
                'di_hints': _di_for_pages(ctx, set(pages)),
            }
            content = [{'type': 'text', 'text': json.dumps(text_payload, separators=(',', ':'))}]
            for p in pages:
                png = _render_page_b64(doc, p - 1, MAPPER_RENDER_DPI)
                content.append({'type': 'image_url',
                                'image_url': {'url': f'data:image/png;base64,{png}',
                                              'detail': 'high'}})

            print(f'  LLM call: pages {pages[0]}-{pages[-1]} ({len(chunk_widgets)} widgets)')
            response = client.chat.completions.parse(
                model=model,
                messages=[{'role': 'system', 'content': MAPPER_SYSTEM_PROMPT},
                          {'role': 'user', 'content': content}],
                response_format=_LLMChunkResponse,
            )
            parsed = response.choices[0].message.parsed
            if parsed is None:
                all_unmapped.extend(w.field_name for w in chunk_widgets); continue
            v, u = _validate_chunk(parsed, chunk_widgets)
            all_bindings += v; all_unmapped += u
    finally:
        doc.close()

    return WidgetMapping(file_name=pdf_path.name, model=model, chunk_count=chunks,
                         bindings=all_bindings,
                         unmapped_widget_field_names=list(dict.fromkeys(all_unmapped)))

print(f'Mapping with {MAPPER_MODEL}...')
mapping = map_widgets(INPUT_PDF, doc_ctx, catalog)
MAPPING_JSON.write_text(mapping.model_dump_json(indent=2))

print(f'Bindings  : {len(mapping.bindings)}')
print(f'Unmapped  : {len(mapping.unmapped_widget_field_names)}')
print(f'Chunks    : {mapping.chunk_count}')
print(f'Saved     : {MAPPING_JSON}')

## 6. Build the `FormSchema`

Turn the widget→label bindings into the generic `FormSchema` shape downstream stages consume. Group bindings by `semantic_field_id`. Radio groups collapse multiple bindings into one `FormField` with an `options` list; everything else becomes one field per widget. `field_type` is derived from the widget type and rect height (tall text widgets become `long_text`). Each `FormField` carries its `widget_bindings` so the writer can find the right widgets later.

In [ ]:
LONG_TEXT_MIN_HEIGHT_PT = 30.0

def _snake(label: str) -> str:
    s = re.sub(r'[^a-zA-Z0-9]+', '_', label.strip().lower()).strip('_')
    return s or 'field'

def _uniqify(field_id: str, taken: set) -> str:
    if field_id not in taken:
        taken.add(field_id); return field_id
    i = 2
    while f'{field_id}_{i}' in taken:
        i += 1
    out = f'{field_id}_{i}'; taken.add(out); return out

def _type_and_options(w: Widget, b: WidgetBinding):
    if w.widget_type == 'checkbox':  return 'checkbox', [b.label_text]
    if w.widget_type == 'choice':    return 'radio_group', list(w.choice_values or [])
    if w.widget_type == 'signature': return 'signature', None
    height = w.rect.y1 - w.rect.y0
    return ('long_text' if height > LONG_TEXT_MIN_HEIGHT_PT else 'text'), None

def build_schema_from_mapping(catalog: WidgetCatalog, mapping: WidgetMapping) -> FormSchema:
    by_xref = {w.xref: w for w in catalog.widgets}
    groups: dict = {}
    for b in mapping.bindings:
        groups.setdefault(b.semantic_field_id, []).append(b)

    taken, fields = set(), []
    for raw_id, group in groups.items():
        widgets = [by_xref.get(b.widget_xref) for b in group]
        widgets = [w for w in widgets if w is not None]
        if not widgets: continue
        types = {w.widget_type for w in widgets}

        if types == {'radio'} and len(group) > 1:
            label = group[0].label_text
            options = [b.option_label or b.on_value or 'Option' for b in group]
            fields.append(FormField(
                field_id=_uniqify(_snake(raw_id), taken), label=label, section=None,
                field_type='radio_group', options=options, format_hint=None,
                required=False, source_text=label, widget_bindings=group,
            ))
        else:
            for b, w in zip(group, widgets):
                ftype, options = _type_and_options(w, b)
                fields.append(FormField(
                    field_id=_uniqify(_snake(b.semantic_field_id), taken),
                    label=b.label_text, section=None, field_type=ftype, options=options,
                    format_hint=None, required=False, source_text=b.label_text,
                    widget_bindings=[b],
                ))
    return FormSchema(form_title=None, issuer=None, sections=[], fields=fields)

schema = build_schema_from_mapping(catalog, mapping)
SCHEMA_JSON.write_text(schema.model_dump_json(indent=2))

print(f'Fields   : {len(schema.fields)}')
print(f'Sections : {len(schema.sections)}')
print(f'Saved    : {SCHEMA_JSON}')

## 7. Persist the mapping under its fingerprint

Write `mappings/<fingerprint>.{mapping,schema,cached}.json` plus an index entry. The next run on this blank form skips Steps 3, 5, 6 entirely.

In [ ]:
CACHE_DIR = REPO_ROOT / 'mappings'

def _atomic_write(path: Path, content: str) -> None:
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(content)
    os.replace(tmp, path)

def store_cached_mapping(fingerprint: str, mapping: WidgetMapping, schema: FormSchema,
                         source_pdf_basename: str) -> None:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    created = datetime.now(timezone.utc).isoformat(timespec='seconds')
    cached = CachedMapping(form_fingerprint=fingerprint, source_pdf_basename=source_pdf_basename,
                           created_at=created, mapping=mapping, form_schema=schema)
    _atomic_write(CACHE_DIR / f'{fingerprint}.cached.json',  cached.model_dump_json(indent=2))
    _atomic_write(CACHE_DIR / f'{fingerprint}.mapping.json', mapping.model_dump_json(indent=2))
    _atomic_write(CACHE_DIR / f'{fingerprint}.schema.json',  schema.model_dump_json(indent=2))

    idx_path = CACHE_DIR / 'index.json'
    index = CacheIndex.model_validate_json(idx_path.read_text()) if idx_path.exists() else CacheIndex(entries=[])
    index.entries = [e for e in index.entries if e.form_fingerprint != fingerprint]
    index.entries.append(CacheIndexEntry(form_fingerprint=fingerprint,
                                         source_pdf_basename=source_pdf_basename,
                                         created_at=created))
    _atomic_write(idx_path, index.model_dump_json(indent=2))

store_cached_mapping(fingerprint, mapping, schema, source_pdf_basename=INPUT_PDF.name)

print(f'Cache dir : {CACHE_DIR}')
for p in sorted(CACHE_DIR.glob(f'{fingerprint}*')):
    print(f'  {p.name}')

## 8. Load the canonical DB JSON

The DB JSON is the **single source of truth** for every form. Its shape is **fixed** across all cases (validated against `DBEnvelope`); only values and `claim.serviceLines` cardinality vary case-to-case.

Strip the envelope (`{success, message, result, executionTimeSec}` → the inner `result`), validate, and show top-level keys.

In [ ]:
def load_db_json(path: Path) -> dict:
    raw = json.loads(path.read_text())
    if not isinstance(raw, dict) or 'result' not in raw:
        raise ValueError(f"{path}: expected an object with a top-level 'result' key")
    result = raw['result']
    if not isinstance(result, dict):
        raise ValueError(f"{path}: 'result' must be an object")
    return result

db = load_db_json(DB_TEMPLATE)
print(f'loaded {DB_TEMPLATE} -> result keys: {list(db.keys())}')
print(f"patient.firstName   = {db['patient']['firstName']!r}")
print(f"claim.claimNumber   = {db['claim']['claimNumber']!r}")
print(f"serviceLines count  = {len(db['claim']['serviceLines'])}")

# Validate against the canonical pydantic model
env_obj = DBEnvelope.model_validate(json.loads(DB_TEMPLATE.read_text()))
print('DBEnvelope validation: OK')

## 8a. (Optional) LLM-fill empty fields in the DB template

If the template has empty strings/lists/`null`s (typical in dev/testing), fill them with a single LLM call against the **fixed** `DBEnvelope` shape. In production this step is skipped — the real DB JSON arrives populated.

In [ ]:
SYSTEM_PROMPT_TEMPLATE_FILL = '''You receive a JSON document representing a healthcare claim
appeal record. Some fields are populated; others are empty strings, empty lists,
or null. Return the SAME document with every empty field filled with realistic,
internally-consistent fictional values.

Rules:
- Do not change any field that already has a non-empty value.
- Keep all field names exactly as given (including any typos in source keys).
- Values must be consistent: same fictional individual, plausible dates, amounts add up.
- Date format: keep the format consistent with whatever other dates use.
  If all dates are empty, use ISO YYYY-MM-DD.
- Service lines: fill empty fields if entries exist, or add 2-4 if empty.
- Never use real personal data.
- Do not invent NEW top-level keys or sub-objects.
'''

def fill_db_template(template: dict, model: str = 'gpt-5-mini') -> dict:
    client = OpenAI(api_key=OPENAI_API_KEY)
    normalized = DBEnvelope.model_validate(template).model_dump()
    user_msg = json.dumps(normalized, indent=2)
    resp = client.chat.completions.parse(
        model=model,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT_TEMPLATE_FILL},
            {'role': 'user',   'content': user_msg},
        ],
        response_format=DBEnvelope,
    )
    parsed = resp.choices[0].message.parsed
    if parsed is None:
        raise RuntimeError(f'empty parse result; finish_reason={resp.choices[0].finish_reason}')
    return parsed.model_dump()

# Detect whether the template needs filling
raw_envelope = json.loads(DB_TEMPLATE.read_text())
_flat_sample = json.dumps(raw_envelope)
needs_fill = '"":' in _flat_sample or '""' in _flat_sample

if needs_fill:
    print('Template has empty fields. Running LLM template-fill...')
    filled_envelope = fill_db_template(raw_envelope)
    DB_FILLED_JSON.parent.mkdir(parents=True, exist_ok=True)
    DB_FILLED_JSON.write_text(json.dumps(filled_envelope, indent=2))
    db = filled_envelope['result']
    print(f'  wrote {DB_FILLED_JSON}')
    print(f"  provider.renderingProviderName = {db['provider']['renderingProviderName']!r}")
else:
    print('Template appears already populated; skipping LLM fill.')

## 9. Canonical DB paths + DB-schema fingerprint

Derive the leaf paths from `DBEnvelope` (pydantic) rather than from any instance, so the fingerprint stays stable even when `claim.serviceLines` happens to be empty in one case and populated in another. List indices are normalized to `[]` so per-row paths collapse to a single canonical form.

In [ ]:
import typing as _t

def canonical_db_paths() -> list:
    def walk(model_cls, prefix: str, out: list) -> None:
        try:
            fields = model_cls.model_fields
        except AttributeError:
            out.append(prefix); return
        for name, info in fields.items():
            key = f'{prefix}.{name}' if prefix else name
            ann = info.annotation
            origin = _t.get_origin(ann)
            args = _t.get_args(ann)
            if origin is list:
                inner = args[0] if args else None
                if inner is not None and hasattr(inner, 'model_fields'):
                    walk(inner, key + '[]', out)
                else:
                    out.append(key + '[]')
            elif hasattr(ann, 'model_fields'):
                walk(ann, key, out)
            else:
                out.append(key)
    paths: list = []
    walk(DBEnvelope, '', paths)
    return sorted(set(
        p[len('result.'):] for p in paths if p.startswith('result.')
    ))

def db_schema_fingerprint() -> str:
    body = '\n'.join(canonical_db_paths())
    return hashlib.sha256(body.encode()).hexdigest()[:16]

ALL_PATHS = canonical_db_paths()
print(f'canonical paths: {len(ALL_PATHS)}')
for p in ALL_PATHS[:12]:
    print(f'  {p}')
print('  ...')
DB_FP = db_schema_fingerprint()
print(f'\nDB schema fingerprint: {DB_FP}')

## 10. Schema fingerprint (form-specific cache key for the fillmap)

Hash the **shape** of the `FormSchema` (field_id, label, section, field_type, sorted options) — not `source_text`. Together with the DB fingerprint this is the key for the resolved `FieldMap` (Step 13). Same form filled for 500 patients = 1 LLM resolution.

In [ ]:
def compute_schema_fingerprint(schema: FormSchema) -> str:
    body = json.dumps(
        [
            dict(
                field_id=f.field_id, label=f.label, section=f.section,
                field_type=f.field_type,
                options=sorted(f.options) if f.options else None,
            )
            for f in schema.fields
        ],
        sort_keys=True,
    )
    return hashlib.sha256(body.encode()).hexdigest()[:16]

SCHEMA_FP = compute_schema_fingerprint(schema)
print(f'schema_fp:           {SCHEMA_FP}')
print(f'db_schema_fp:        {DB_FP}')
print(f'fillmap cache file:  {FILLMAP_JSON}')

## 11. LLM resolve — structured-output batch over every field

Send every form field, the canonical path list, and a small set of sample DB values to `gpt-5-mini` in a single call. The response validates against `LLMResolverResponse` (pydantic). The model picks `kind` per field: `scalar`, `compose` (template + multiple paths), `service_line` (table row), `derived` (predicate), `constant`, or `unmapped`. Choice fields get one `OptionResolution` per option.

One LLM call per form, regardless of how many cases (`FieldMap` cached next step).

In [ ]:
LLM_SYSTEM_PROMPT = '''You receive a list of form fields and the FLATTENED list
of available DB paths (dotted notation; list indices are normalized to "[]").

For each form field, produce a FieldResolution. Allowed `value.kind` values:
- "scalar"       : single DB path supplies the value.
- "compose"      : combine multiple paths via a Python str.format template
                   such as "{0} {1} {2}".
- "service_line" : the field belongs to a table row; use one path with the
                   literal "[{i}]" placeholder where the row index goes, and
                   set `service_line_index` to the 0-based row index inferred
                   from the field_id or label.
- "derived"      : the value is a small predicate over the DB (eq, neq,
                   non_empty, empty). Use `derive` with op/path/value and
                   `true_value` / `false_value`.
- "constant"     : literal value (rare; use only when no DB path applies but
                   a stable constant should be stamped).
- "unmapped"     : no DB path plausibly supplies the value.

For radio_group / checkbox fields:
- Set `value.kind` to "constant" with literal=null.
- For each option in the field's `options` list, emit one OptionResolution
  whose `when` is a Resolution that determines selection. Use
  `selected_if="equals"` with `equals_value` for derived resolutions, or
  `"truthy"` to select when the resolved value is non-empty.

Rules:
- NEVER invent paths not in the provided list.
- NEVER reference field labels, section names, or option values from any
  specific form in your reasoning. Work only from what you are given.
- Apply `transform` when the field_type clearly requires coercion:
  "date:<out_format>" (use the field's format_hint as out_format if given;
  default "YYYY-MM-DD"), "phone:formatted", "phone:digits", "number",
  "currency", "upper", "lower", "strip".
- Confidence: "high" for unambiguous matches, "medium" for plausible,
  "low" for guesses, "unmapped" if no plausible path exists.
- Set `reason` to one short sentence explaining the choice.

Return every input field with exactly one FieldResolution, in the same order.
'''

def _field_summary(f: FormField) -> dict:
    return dict(
        field_id=f.field_id, label=f.label, section=f.section,
        field_type=f.field_type, options=f.options,
        format_hint=f.format_hint, required=f.required,
    )

def _flatten_db(data: Any, prefix: str = '') -> dict:
    out: dict = {}
    if isinstance(data, dict):
        for k, v in data.items():
            key = f'{prefix}.{k}' if prefix else k
            out.update(_flatten_db(v, key))
    elif isinstance(data, list) and data:
        for i, item in enumerate(data):
            out.update(_flatten_db(item, f'{prefix}[{i}]'))
    else:
        out[prefix] = data
    return out

def _normalize_path(path: str) -> str:
    out, i = '', 0
    while i < len(path):
        if path[i] == '[':
            j = path.index(']', i); out += '[]'; i = j + 1
        else:
            out += path[i]; i += 1
    return out

def _sample_values(db: dict, n: int = 60) -> dict:
    flat = _flatten_db(db)
    seen: dict = {}
    for k, v in flat.items():
        if v in ('', None, [], {}): continue
        norm = _normalize_path(k)
        if norm in seen: continue
        seen[norm] = v
        if len(seen) >= n: break
    return seen

def llm_resolve(fields: list, paths: list, db: dict, model: str = 'gpt-5-mini') -> list:
    if not fields: return []
    client = OpenAI(api_key=OPENAI_API_KEY)
    user_msg = json.dumps(dict(
        form_fields=[_field_summary(f) for f in fields],
        db_paths=paths,
        db_sample_values=_sample_values(db),
    ), indent=2)
    resp = client.chat.completions.parse(
        model=model,
        messages=[
            {'role': 'system', 'content': LLM_SYSTEM_PROMPT},
            {'role': 'user',   'content': user_msg},
        ],
        response_format=LLMResolverResponse,
    )
    parsed = resp.choices[0].message.parsed
    if parsed is None:
        raise RuntimeError(f'empty parse; finish_reason={resp.choices[0].finish_reason}')
    by_id = {fr.field_id: fr for fr in parsed.field_resolutions}
    return [
        by_id.get(f.field_id, FieldResolution(
            field_id=f.field_id, confidence='unmapped',
            value=Resolution(kind='unmapped', reason='missing from LLM response'),
        ))
        for f in fields
    ]

field_resolutions = llm_resolve(schema.fields, ALL_PATHS, db)
print(f"LLM resolved: {sum(1 for fr in field_resolutions if fr.confidence != 'unmapped')}/{len(field_resolutions)}")
print()
for fr in field_resolutions[:8]:
    p = fr.value.paths[0] if fr.value.paths else '-'
    print(f'  {fr.field_id:35s} kind={fr.value.kind:12s} conf={fr.confidence:8s} -> {p}')

## 12. Persist the `FieldMap` (form-keyed cache)

Write the resolution to `<stem>.fillmap.json`. On the next run with the same form + same DB schema fingerprint, the resolver LLM call is skipped entirely (load the cached FieldMap and go straight to materialize).

In [ ]:
fmap = FieldMap(
    schema_fingerprint=SCHEMA_FP,
    db_schema_fingerprint=DB_FP,
    form_pdf_basename=STEM,
    created_at=datetime.now(timezone.utc).isoformat(),
    field_resolutions=field_resolutions,
    unresolved_field_ids=[fr.field_id for fr in field_resolutions if fr.confidence == 'unmapped'],
    notes=[
        f"LLM resolved {sum(1 for fr in field_resolutions if fr.confidence != 'unmapped')}/{len(field_resolutions)}",
    ],
)

FILLMAP_JSON.write_text(fmap.model_dump_json(indent=2))
print(f'wrote {FILLMAP_JSON}')
by_kind: dict = {}
for fr in field_resolutions:
    by_kind[fr.value.kind] = by_kind.get(fr.value.kind, 0) + 1
print(f'resolution kinds:  {by_kind}')
print(f'unresolved fields: {fmap.unresolved_field_ids}')

## 13. Materialize — `FieldMap` + DB → `FilledFormSchema`

Deterministic application. No LLM, no network. For every `FormField`: look up its `FieldResolution`, resolve the raw value per `kind`, apply the `transform`, accumulate selected options for choice fields. `widget_bindings` carries through unchanged so the writer can find the right widgets.

In [ ]:
_DATE_PATTERNS = [
    ('%Y%m%d',   r'^\d{8}$'),
    ('%m/%d/%Y', r'^\d{1,2}/\d{1,2}/\d{4}$'),
    ('%m-%d-%Y', r'^\d{1,2}-\d{1,2}-\d{4}$'),
    ('%Y-%m-%d', r'^\d{4}-\d{1,2}-\d{1,2}$'),
]
_DATE_OUT = {
    'YYYY-MM-DD': '%Y-%m-%d', 'MM/DD/YYYY': '%m/%d/%Y',
    'MM-DD-YYYY': '%m-%d-%Y', 'YYYYMMDD':   '%Y%m%d',
    'DD/MM/YYYY': '%d/%m/%Y',
}

def coerce_date(raw: str, out_format=None):
    if not raw or not isinstance(raw, str): return None
    s = raw.strip()
    if not s: return None
    parsed = None
    for fmt, pat in _DATE_PATTERNS:
        if re.match(pat, s):
            try: parsed = datetime.strptime(s, fmt); break
            except ValueError: continue
    if parsed is None: return None
    return parsed.strftime(_DATE_OUT.get(out_format or 'YYYY-MM-DD', '%Y-%m-%d'))

def coerce_phone(raw: str, style: str = 'formatted'):
    if not raw or not isinstance(raw, str): return None
    digits = re.sub(r'\D', '', raw)
    if len(digits) < 7: return None
    if style == 'digits': return digits
    if len(digits) == 10: return f'({digits[0:3]}) {digits[3:6]}-{digits[6:]}'
    if len(digits) == 11 and digits[0] == '1':
        return f'+1 ({digits[1:4]}) {digits[4:7]}-{digits[7:]}'
    return digits

def coerce_number(raw):
    if raw is None or raw == '': return None
    s = str(raw).replace(',', '').strip()
    try: f = float(s)
    except ValueError: return None
    return str(int(f)) if f == int(f) else f'{f}'

def coerce_currency(raw):
    if raw is None or raw == '': return None
    s = str(raw).replace('$', '').replace(',', '').strip()
    try: f = float(s)
    except ValueError: return None
    return f'{f:,.2f}'

def apply_transform(raw, transform):
    if raw is None or raw == '': return None
    if transform is None: return str(raw)
    if transform.startswith('date:'):
        return coerce_date(str(raw), out_format=transform.split(':', 1)[1] or None)
    if transform == 'phone:digits':            return coerce_phone(str(raw), style='digits')
    if transform in ('phone', 'phone:formatted'): return coerce_phone(str(raw), style='formatted')
    if transform == 'number':                  return coerce_number(str(raw))
    if transform == 'currency':                return coerce_currency(str(raw))
    if transform == 'upper':                   return str(raw).upper()
    if transform == 'lower':                   return str(raw).lower()
    if transform == 'strip':                   return str(raw).strip()
    raise ValueError(f'Unknown transform: {transform!r}')

def compose(values: list, template: str):
    stringified = ['' if v is None else str(v) for v in values]
    rendered = re.sub(r'\s+', ' ', template.format(*stringified)).strip()
    return rendered if rendered else None

def db_path_get(data: dict, dotted: str):
    parts: list = []; buf = ''; i = 0
    while i < len(dotted):
        ch = dotted[i]
        if ch == '.':
            if buf: parts.append(buf); buf = ''
            i += 1
        elif ch == '[':
            if buf: parts.append(buf); buf = ''
            j = dotted.index(']', i)
            parts.append(int(dotted[i+1:j])); i = j + 1
        else:
            buf += ch; i += 1
    if buf: parts.append(buf)
    cur = data
    for p in parts:
        if cur is None: return None
        if isinstance(p, int):
            if not isinstance(cur, list) or p >= len(cur): return None
            cur = cur[p]
        else:
            if not isinstance(cur, dict) or p not in cur: return None
            cur = cur[p]
    return cur

def resolve_value(r: Resolution, db: dict):
    if r.kind == 'constant':    return r.literal
    if r.kind == 'unmapped':    return None
    if r.kind == 'scalar':      return db_path_get(db, r.paths[0]) if r.paths else None
    if r.kind == 'compose':
        raws = [db_path_get(db, p) for p in r.paths]
        return compose(raws, r.template or '')
    if r.kind == 'service_line':
        idx = r.service_line_index or 0
        if not r.paths: return None
        path = r.paths[0].replace('{i}', str(idx)).replace('[]', f'[{idx}]')
        return db_path_get(db, path)
    if r.kind == 'derived':
        d = r.derive
        if d is None: return None
        observed = db_path_get(db, d.path)
        if d.op == 'eq':         matched = str(observed) == str(d.value or '')
        elif d.op == 'neq':      matched = str(observed) != str(d.value or '')
        elif d.op == 'non_empty': matched = observed not in (None, '')
        elif d.op == 'empty':    matched = observed in (None, '')
        else: raise ValueError(f'Unknown derived op: {d.op!r}')
        return d.true_value if matched else d.false_value
    raise ValueError(f'Unknown Resolution.kind: {r.kind!r}')

def option_matches(opt: OptionResolution, db: dict) -> bool:
    val = resolve_value(opt.when, db)
    if opt.selected_if == 'always':  return val is not None and val != ''
    if opt.selected_if == 'equals':  return str(val) == (opt.equals_value or '')
    return val is not None and val != '' and val is not False

def fill_from_db(schema: FormSchema, fmap: FieldMap, db: dict) -> FilledFormSchema:
    by_id = {fr.field_id: fr for fr in fmap.field_resolutions}
    filled: list = []
    for f in schema.fields:
        fr = by_id.get(f.field_id)
        if fr is None:
            filled.append(FilledFormField(**f.model_dump(), value=None)); continue
        if f.field_type in ('checkbox', 'radio_group'):
            sel = [o.option_label for o in fr.options if option_matches(o, db)]
            v = (sel[0] if sel else None) if f.field_type == 'radio_group' else (sel if sel else None)
        else:
            raw = resolve_value(fr.value, db)
            v = apply_transform(raw, fr.value.transform) if raw is not None else None
        filled.append(FilledFormField(**f.model_dump(), value=v))
    return FilledFormSchema.from_schema(schema, filled)

filled = fill_from_db(schema, fmap, db)
FILLED_JSON.write_text(filled.model_dump_json(indent=2))
print(f'wrote {FILLED_JSON}\n')
for f in filled.fields:
    v = f.value
    if isinstance(v, list): vs = repr(v)
    elif v is None:         vs = '(empty)'
    else:                   vs = str(v)[:70]
    print(f'  {f.field_id:55s} -> {vs}')

## 14. Write the filled PDF

Hand the `FilledFormSchema` to `write_mapped` (`claims_parser.mapped_writer`). Each `FilledFormField` already carries its `widget_bindings` (from Step 6), so the writer locates the widget by xref, sets `widget.field_value`, calls `widget.update()`, and records any field it couldn't stamp in the review report. Pass `flatten=True` to bake widgets into static content; default is editable output.

In [ ]:
review = write_mapped(
    pdf_path=INPUT_PDF,
    filled=filled,
    output_path=FILLED_PDF,
    flatten=False,
)
REVIEW_JSON.write_text(review.model_dump_json(indent=2))
print(f'wrote {FILLED_PDF}')
print(f'wrote {REVIEW_JSON}')

counts = review.by_reason()
if counts:
    print('\nReview flags by reason:')
    for k, v in counts.items():
        print(f'  {k:30s} {v}')
else:
    print('\nNo fields flagged for review.')

## 15. Summary

What was produced, where it lives, and what runs next time the same form is filled for a different case.

In [ ]:
print('Per-run artifacts:')
for p in [WIDGETS_JSON, CONTEXT_JSON, MAPPING_JSON, SCHEMA_JSON,
          FILLMAP_JSON, FILLED_JSON, REVIEW_JSON]:
    if p.exists():
        print(f'  {p}  ({p.stat().st_size / 1024:.1f} KB)')
print(f'  {FILLED_PDF}  ({FILLED_PDF.stat().st_size / 1024:.1f} KB)')

total = len(filled.fields)
filled_n = sum(1 for f in filled.fields if f.value not in (None, '', []))
unmapped = len(fmap.unresolved_field_ids)
print(f'\nFields:  total={total}  filled={filled_n}  unmapped={unmapped}')

print('\nCache reuse:')
print(f'  widget fingerprint    = {fingerprint}')
print(f'  schema_fingerprint    = {SCHEMA_FP}')
print(f'  db_schema_fingerprint = {DB_FP}')
print(f'  next run on the same form -> skips Steps 3, 5, 6, 11 (vision-LLM + resolver)')
print(f'  next run on a new case for the same form -> 0 LLM calls; just re-materialize')